# Module 14 — MCP + SDK (search / read)

Exposer le **même** corpus au chat *et* à un agent, sans second index.

**Prérequis** : OpenSearch + Qdrant + MinIO ; silver indexé (`presslake pipeline`).

Tuto : [`docs/modules/14-mcp-sdk.md`](../docs/modules/14-mcp-sdk.md) · ADR [0008](../docs/adr/0008-mcp-search-read.md)

## Cas A — Le serveur déclare deux outils

In [3]:
from presslake.mcp.server import create_server

# Jupyter a déjà une event loop : await, pas asyncio.run()
tools = await create_server().list_tools()
names = sorted(t.name for t in tools)
print(names)
assert names == ["read", "search"]


['read', 'search']


## Cas B — `search` = retrieve hybride (pas de LLM)

In [4]:
import json

from presslake.mcp.tools import search_corpus

raw = search_corpus("Népal", limit=3)
data = json.loads(raw)
print("count", data.get("count"))
if data.get("passages"):
    p0 = data["passages"][0]
    print("hash", p0.get("content_hash"))
    print("silver", p0.get("silver_s3_uri"))
    print((p0.get("text") or "")[:200])

/home/anthony-marais/Documents/data_project/src/presslake/vector/embed.py:13: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  return TextEmbedding(model_name=embedding_model())


count 3
hash feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1
silver s3://presslake/silver/source=france24/dt=2026-08-31/feb162e21ec643072a8da0daf1034cf458c5479cfef5ab5ddd9d8d75d6c7a0b1.json
Crise au <em>Népal</em> : onde de choc régionale ?


## Cas C — `read` refuse le bronze

In [5]:
from presslake.mcp.tools import read_silver
import json

denied = json.loads(
    read_silver(silver_s3_uri="s3://presslake/bronze/source=x/dt=2026-01-01/abc.json")
)
print(denied)
assert "error" in denied
assert "silver" in denied["error"] or "bronze" in denied["error"].lower() or "refuse" in denied["error"]

{'error': "read refuse le bronze et tout chemin hors silver/ — uri='s3://presslake/bronze/source=x/dt=2026-01-01/abc.json'", 'silver_s3_uri': 's3://presslake/bronze/source=x/dt=2026-01-01/abc.json'}


## Cas D — `read` un silver (si search a renvoyé une URI)

In [6]:
uri = None
digest = None
if data.get("passages"):
    uri = data["passages"][0].get("silver_s3_uri")
    digest = data["passages"][0].get("content_hash")

if not uri and not digest:
    print("Pas de hit — lancer uv run presslake pipeline puis rejouer cette cellule.")
else:
    doc = json.loads(read_silver(content_hash=digest, max_chars=500))
    print("keys", sorted(doc.keys()))
    print((doc.get("text") or doc.get("error") or "")[:400])

keys ['canonical_url', 'content_hash', 'content_lang', 'feed_id', 'silver_s3_uri', 'text', 'text_source', 'title', 'truncated']
Crise au Népal : onde de choc régionale ?
Pour afficher ce contenu YouTube, il est nécessaire d'autoriser les cookies de mesure d'audience et de publicité.
Une extension de votre navigateur semble bloquer le chargement du lecteur vidéo. Pour pouvoir regarder ce contenu, vous devez la désactiver ou la désinstaller.
Publié le :
Au Népal, le bilan des victimes est revu à la hausse chaque jour. Des id


## Cas E — Cursor

`.cursor/mcp.json` → serveur `presslake` → `uv run presslake mcp`.

Ne lance **pas** `presslake mcp` dans ce notebook (stdio bloquant).